# Native dataset inputs, checking and encoding

Start after source-specific conversion to native conversations. Choose source branches, system settings, tool-result associations and loss weights before running the full check. The legacy posttraining-data stages produce other schemas; their output still needs an explicit adapter. Already-native data can start here directly.

Native JSON can be stored one record per JSONL line or as a JSON string column. Loading preserves message order and payloads, including malformed tool-argument text. The standalone integration reads Parquet or saved HF datasets with a `conversation_json` string column and owns corpus iteration, reports, workers and resume. See `apertus2-processing/README.md` in [posttraining-data](https://github.com/swiss-ai/posttraining-data) for its `prepare`, `encode` and `merge` commands.


In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory

from pydantic import ValidationError

from apertus_common import (
    Conversation,
    Document,
    Host,
    OpenAIMessage,
    Reply,
    SystemPrompt,
    User,
    Wait,
    checkers_for,
    from_openai,
    to_openai,
)

system = SystemPrompt.build(behavior="Be concise.")
records = [
    Conversation(
        system=system, items=[User(payload="Hi"), Reply(payload="Hello"), Wait()]
    ),
    Conversation(
        system=system, items=[User(payload="First"), Host(payload="Later input")]
    ),
]

with TemporaryDirectory(prefix="apertus-dataset-") as temporary:
    path = Path(temporary) / "conversations.jsonl"
    path.write_text(
        "".join(record.to_json() + "\n" for record in records), encoding="utf-8"
    )
    restored: list[Conversation] = []
    findings: Counter[str] = Counter()
    checks = checkers_for("full")  # Reuse compiled schemas across records.
    with path.open(encoding="utf-8") as source:
        for number, line in enumerate(source, start=1):
            try:
                record = Conversation.from_json(line)
            except ValidationError as error:
                print(f"Line {number}: {error}")
                continue
            report = record.check(checkers=checks)
            findings.update(finding.rule for finding in report.violations)
            findings.update(
                f"unevaluated:{finding.rule}" for finding in report.unevaluated
            )
            restored.append(record)
    assert restored == records
    assert not findings
    assert [item.type for item in restored[1].items] == ["user", "host"]

## Exact native and wrapped input layouts

A native conversation uses `schema_version`, `system` and ordered `items`. It is distinct from OpenAI role/content messages and rendered control-token text. Complete-history checking needs a system prompt with a thinking setting. Waits carry `type` only, plus optional loss metadata.

The library examples below demonstrate bare and wrapped JSONL storage. Bare JSONL stores the native object on one physical line; wrapped JSONL stores its serialized JSON string in `conversation_json`, alongside metadata such as `record_id`. For the standalone processing CLI, save that string column in Parquet or an HF `save_to_disk()` dataset instead; JSONL is not a CLI input. Use `--column NAME` to select another column. A nested conversation object is not accepted in that column.


In [ ]:
import json

native_record = Conversation.from_json(
    '{"schema_version":2,"system":{"thinking":"medium","tools":[]},'
    '"items":[{"type":"user","payload":"Hello"},'
    '{"type":"reply","payload":"Hi"},{"type":"wait"}]}'
)
native_report = native_record.check()
assert native_report.ok and native_report.complete
assert native_record.items[-1].to_json() == '{"type":"wait"}'

wrapped_record = {
    "record_id": "example-1",
    "conversation_json": native_record.to_json(),
}
wrapped_line = json.dumps(wrapped_record, ensure_ascii=False)
loaded_wrapper = json.loads(wrapped_line)
assert isinstance(loaded_wrapper["conversation_json"], str)
assert Conversation.from_json(loaded_wrapper["conversation_json"]) == native_record
# An HF row has these same fields. Use save_to_disk(), then pass that directory.

## Documents require provenance

The library prepares post-training data. Every document declares its supplying party (`from`) and citation ID (`id`); records without them fail the check.

In [ ]:
missing = Conversation(system=system, items=[Document(payload="Source text")])
assert not missing.check().ok
cited = Conversation(
    system=system,
    items=[Document(payload="Source text", from_="user", id="d1")],
)
report = cited.check()
assert report.ok and report.complete
assert Conversation.from_json(cited.to_json()) == cited

## OpenAI mappings return values, diagnostics, and context

Use `.value` for the mapped data. Preserve `.context` when continuing tool conversations or compacting history. The default mapper rejects unsupported or lossy conversions; explicit policies allow particular documented losses.

See [the focused conversion notebook](openai_conversion_examples.ipynb) for normalizations, lossy policies, rejected mappings and training/inference consequences.


In [ ]:
messages: list[OpenAIMessage] = [
    {"role": "system", "content": "Be concise."},
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi"},
]
mapped = from_openai(messages, reasoning_effort="medium")
assert mapped.value.check().ok
assert not mapped.diagnostics
request = to_openai(mapped.value, context=mapped.context)
assert request.value.get("messages") == messages
assert request.value.get("reasoning_effort") == "medium"
assert Conversation.from_json(mapped.value.to_json()) == mapped.value
print(request.value)

Keeping native conversation JSON in a text column avoids imposing an external table schema on the message union. OpenAI conversion is a supported subset, not a replacement for native storage. Tokenizer artifacts are acquired separately and are unnecessary for JSONL processing.


## Bounded batch encoding and loss weights

Acquire the instruct tokenizer separately and set `APERTUS_ENCODING` to its local directory before running this section. The library batches ordinary text through the tokenizer backend without changing token IDs. Bound batches in the caller and initialize one encoding/checker set per worker. The standalone posttraining-data integration owns dataset shards, worker pools, reports and resume.

Optional message weights are native v2 metadata. Default checking rejects them; opt in through reusable checkers. Missing output and wait weights default to 1; missing input/system weights default to 0. Input frames and headers always receive 0, even when an input payload is weighted. Output weights cover the full frame; waits have their own weight. Returned weights align with token IDs before the training target shift and are not necessarily binary masks.


In [ ]:
import os
from itertools import batched

from apertus_common import load_encoding, render_str

weighted_record = (
    Conversation.builder()
    .system(SystemPrompt.build())
    .user("What is 6 times 7?")
    .think("Multiply the numbers.", loss_weight=0)
    .reply("42", loss_weight=0.5)
    .wait(loss_weight=1)
    .build()
)
assert not weighted_record.check().ok
weighted_checks = checkers_for("full", allow_loss_weights=True)
weighted_report = weighted_record.check(checkers=weighted_checks)
assert weighted_report.ok and weighted_report.complete
assert "loss_weight" not in render_str(weighted_record)
assert Conversation.from_json(weighted_record.to_json()) == weighted_record

artifact = os.environ.get("APERTUS_ENCODING")
if artifact:
    encoding = load_encoding(artifact)
    for batch in batched([*records, weighted_record], 2, strict=False):
        token_batches = encoding.encode_conversations(batch)
        weighted_batches = encoding.encode_conversations(
            batch, return_loss_weights=True
        )
        for native, ids, encoded_weights in zip(
            batch, token_batches, weighted_batches, strict=True
        ):
            assert ids == encoding.encode_conversation(native)
            assert ids == encoded_weights.token_ids
            assert len(ids) == len(encoded_weights.loss_weights)
            assert encoded_weights.loss_weights[0] == 0  # BOS
            print(len(ids), sum(encoded_weights.loss_weights))
else:
    print("Set APERTUS_ENCODING to run the tokenizer-backed batch example.")